# Cox生存分析

* `mydir`：自己的数据
* `ostime_column`: 数据对应的生存时间，不一定非的是OST，也可以是DST、FST等。
* `os`：生存状态，不一定非的是OS，也可以是DS、FS等。

In [22]:
from lifelines import CoxPHFitter
import pandas as pd
from onekey_algo.custom.components.comp1 import normalize_df
from sklearn.model_selection import train_test_split
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.comp1 import fillna

def get_prediction(mn):
    prediction = pd.concat([pd.read_csv(f'results/{mn}_cox_predictions_{subset}.csv') for subset in get_param_in_cwd('subsets')],
                           axis=0)
    prediction.columns = ['ID', f'{mn}_Exp', mn]
    return prediction[['ID', f'{mn}']]

data = None
for mn in get_param_in_cwd('compare_models'):
    pred = get_prediction(mn)
    pred['ID'] = pred['ID'].astype(str)
    if data is None:
        data = pred
    else:
        data = pd.merge(data, pred, on='ID', how='left')

# data = normalize_df(data, not_norm=['ID', 'group'])
label_data = pd.read_csv(get_param_in_cwd('label_file'))#.drop_duplicates('ID')
# label_data = pd.merge(label_data, pd.read_csv('group.csv'), on='ID', how='inner')
# label_data = fillna(label_data, fill_mod='50%')
data = pd.merge(data, label_data, on='ID', how='inner')
data.to_csv('results/joinit_info.csv', index=False)
data

,ID,Clinical,Pathomics,Combined,event,duration,group
0,J07A3054,138.600,131.529,139.370,0,132,train
1,J07A3055,130.586,137.326,139.346,0,131,train
2,J07A3056,120.398,137.054,136.351,1,78,train
3,J07A3057,124.339,137.326,137.676,0,131,train
4,J07A3060,114.160,135.397,133.272,0,130,train
...,...,...,...,...,...,...,...
273,J07A1132,134.498,136.331,139.953,0,108,test
274,J07A1133,130.586,132.938,137.749,0,107,test
275,J07A1062,132.666,136.677,139.635,0,119,test
276,J07A1080,135.418,136.273,140.143,0,115,test


In [23]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.utils import concordance_index
import numpy as np

metrics = []
for mn in get_param_in_cwd('compare_models'):
    metric = []
    for subset in get_param_in_cwd('subsets'):
        subdata = data[data['group'] == subset]
        metric.append(concordance_index(subdata[get_param_in_cwd('duration_col')], subdata[mn],
                                        subdata[get_param_in_cwd('event_col')]))
    metrics.append(metric)
metrics = pd.DataFrame(np.array(metrics).T, columns=get_param_in_cwd('compare_models'))
metrics['Cohort'] = get_param_in_cwd('subsets')
metrics

,Clinical,Pathomics,Combined,Cohort
0,0.683,0.767,0.834,train
1,0.807,0.765,0.832,test
